In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import joblib

# 1. Load Dataset

In [2]:
df = pd.read_csv("../data/gated_delivery_dataset.csv")
df.head()

,Agent_Age,Agent_Rating,Order_Date,Order_Time,Pickup_Time,Weather,Traffic,Vehicle,Area,Delivery_Time,...,visitor_pass_status,society_security_level,gate_wait_time,driver_status,previous_failed_deliveries,address_confidence,preferred_delivery_slot,estimated_arrival_delay,driver_experience,delivery_failure
0,37,4.9,2022-03-19,11:30:00,11:45:00,Sunny,High,motorcycle,Urban,120,...,Approved,Moderate,1,Delayed,1,78,Evening,12,9,0
1,34,4.5,2022-03-25,19:45:00,19:50:00,Stormy,Jam,scooter,Metropolitian,165,...,Pending,Moderate,15,Available,4,87,Evening,3,8,1
2,23,4.4,2022-03-19,08:30:00,08:45:00,Sandstorms,Low,motorcycle,Urban,130,...,Approved,Moderate,3,Available,0,67,Morning,2,5,0
3,38,4.7,2022-04-05,18:00:00,18:10:00,Sunny,Medium,motorcycle,Metropolitian,105,...,Approved,Moderate,1,Available,3,56,Morning,7,6,0
4,32,4.6,2022-03-26,13:30:00,13:45:00,Cloudy,High,scooter,Metropolitian,150,...,Approved,Strict,3,Available,0,87,Afternoon,2,3,0


# 2. Date Formatting & Missing Value Handling (Dates)

In [3]:
# Order Date
df["Order_Date"] = pd.to_datetime(df["Order_Date"], errors="coerce", dayfirst=True)
# Order Time
df["Order_Time"] = pd.to_datetime(df["Order_Time"], errors="coerce", format="mixed")
# Pickup Time
df["Pickup_Time"] = pd.to_datetime(df["Pickup_Time"], errors="coerce", format="mixed")

# Drop rows with invalid date/time values (BEFORE train test split)
df.dropna(subset=["Order_Date", "Order_Time", "Pickup_Time"], inplace=True)

C:\Users\VIDWAN\AppData\Local\Temp\ipykernel_17968\3294361230.py:2: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df["Order_Date"] = pd.to_datetime(df["Order_Date"], errors="coerce", dayfirst=True)


# 3. Feature Engineering

In [4]:
# Pickup delay
df["pickup_delay_minutes"] = ((df["Pickup_Time"] - df["Order_Time"]).dt.total_seconds() / 60)
# Hour of day
df["hour_of_day"] = df["Order_Time"].dt.hour
# Day of week
df["day_of_week"] = df["Order_Date"].dt.day_name()
# Weekend
df["is_weekend"] = df["Order_Date"].dt.weekday >= 5

# Arrival within preferred slot
def arrival_slot(hour):
    if hour < 12:
        return "Morning"
    elif hour < 17:
        return "Afternoon"
    else:
        return "Evening"

df["actual_delivery_slot"] = df["hour_of_day"].apply(arrival_slot)
df["arrival_within_preferred_slot"] = (df["actual_delivery_slot"] == df["preferred_delivery_slot"]).astype(int)

# Customer Reachability Score
df["customer_reachability_score"] = np.where(
    df["customer_answered_call"] == "Yes",
    100 - (df["customer_response_time"] / 6),
    10
)
df["customer_reachability_score"] = df["customer_reachability_score"].clip(0, 100)

# Society Accessibility Score
security_map = {"Open": 100, "Moderate": 70, "Strict": 40}
df["society_accessibility_score"] = df["society_security_level"].map(security_map).fillna(0) - df["gate_wait_time"] * 2
df["society_accessibility_score"] = df["society_accessibility_score"].clip(0, 100)

# Driver Reliability Score
driver_map = {"Available": 100, "Delayed": 60, "Accident": 20, "Medical Emergency": 10}
df["driver_reliability_score"] = (
    df["driver_status"].map(driver_map).fillna(0)
    + df["driver_experience"] * 2
    - df["previous_failed_deliveries"] * 5
)
df["driver_reliability_score"] = df["driver_reliability_score"].clip(0, 100)

# 4. Drop Unnecessary Columns

In [5]:
columns_to_drop = ["Order_Date", "Order_Time", "Pickup_Time", "actual_delivery_slot", "Order_ID", "driver_id", "customer_id"]
for col in columns_to_drop:
    if col in df.columns:
        df.drop(columns=col, inplace=True)

# 5. Handle Missing Values
Use median for numerical columns and mode for categorical columns BEFORE encoding and splitting.

In [6]:
# Identify numerical and categorical columns
numeric_cols = df.select_dtypes(include=["int64", "float64"]).columns
categorical_cols = df.select_dtypes(include=["object"]).columns

# Handle numerical missing values
for col in numeric_cols:
    if col != "delivery_failure":
        df[col] = df[col].fillna(df[col].median())

# Handle categorical missing values
for col in categorical_cols:
    if col != "delivery_failure":
        df[col] = df[col].fillna(df[col].mode()[0])

print("Missing values handled.")

Missing values handled.


# 6. Encode Categorical Columns

In [7]:
label_encoders = {}
for col in categorical_cols:
    if col != "delivery_failure":
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col].astype(str))
        label_encoders[col] = le

print("Categorical Encoding Completed.")

Categorical Encoding Completed.


# 7. Train Test Split & Save

In [8]:
# Ensure target variable is correct
X = df.drop(columns=["delivery_failure"])
y = df["delivery_failure"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print("Training Features :", X_train.shape)
print("Testing Features  :", X_test.shape)
print("Training Labels   :", y_train.shape)
print("Testing Labels    :", y_test.shape)

# Save Processed Data
X_train.to_csv("../data/X_train.csv", index=False)
X_test.to_csv("../data/X_test.csv", index=False)
y_train.to_csv("../data/y_train.csv", index=False)
y_test.to_csv("../data/y_test.csv", index=False)

# Save Label Encoders
joblib.dump(label_encoders, "../models/label_encoders.pkl")

print("Preprocessing Completed Successfully.")

Training Features : (34918, 27)
Testing Features  : (8730, 27)
Training Labels   : (34918,)
Testing Labels    : (8730,)
Preprocessing Completed Successfully.


Preprocessing Completed Successfully.


In [9]:
print(X_train.shape)
print(y_train.shape)

print(X_test.shape)
print(y_test.shape)

(34918, 27)
(34918,)
(8730, 27)
(8730,)
